<a href="https://colab.research.google.com/github/Priyanka777444/unisign-sign-language-recognition/blob/main/Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Image Classification**

In [ ]:
import tensorflow as tf

from keras._tf_keras.keras.layers import Dense, Conv2D ,MaxPooling2D , Flatten ,Dropout , GlobalAveragePooling2D
from keras._tf_keras.keras.models import Sequential
from keras._tf_keras.keras.models import save_model
from keras._tf_keras.keras.preprocessing.image import ImageDataGenerator
from keras._tf_keras.keras.callbacks import EarlyStopping
from keras._tf_keras.keras.optimizers import Adam
from keras._tf_keras.keras.applications import MobileNet

img_width, img_height = 224, 224

from google.colab import drive
drive.mount('/content/drive')



train_data_dir = '/content/drive/MyDrive/Hackathon/DataSet/train_data'
validation_data_dir =train_data_dir
test_data_dir = '/content/drive/MyDrive/Hackathon/DataSet/test_data'
batch_size = 32
base_model = MobileNet(weights='imagenet', include_top=False, input_shape=(img_width, img_height, 3))

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(34, activation='softmax')
])
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)
validation_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_data_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

validation_generator = train_datagen.flow_from_directory(
    validation_data_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

epochs = 51
model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // batch_size,
    epochs=epochs,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // batch_size
)
model.save('image_classification_model.h5')

# test_data = train_datagen.flow_from_directory(
#     test_data_dir,
#     target_size=(img_width, img_height),
#     batch_size=batch_size,
#     class_mode='categorical'
#    )

loss, accuracy = model.evaluate(validation_generator)
print(f'Testing accuracy: {accuracy*100:.2f}%')


Mounted at /content/drive
17225924/17225924 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Found 340 images belonging to 34 classes.
Found 68 images belonging to 34 classes.
Epoch 1/51


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


10/10 ━━━━━━━━━━━━━━━━━━━━ 124s 4s/step - accuracy: 0.4143 - loss: 2.5666 - val_accuracy: 0.0469 - val_loss: 7.0413
Epoch 2/51
 1/10 ━━━━━━━━━━━━━━━━━━━━ 1s 112ms/step - accuracy: 0.9688 - loss: 0.1444

/usr/lib/python3.10/contextlib.py:153: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(typ, value, traceback)


10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 267ms/step - accuracy: 0.9688 - loss: 0.1444 - val_accuracy: 0.0000e+00 - val_loss: 6.1663
Epoch 3/51
10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 378ms/step - accuracy: 0.9837 - loss: 0.0893 - val_accuracy: 0.2031 - val_loss: 5.8452
Epoch 4/51
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 114ms/step - accuracy: 1.0000 - loss: 0.0374 - val_accuracy: 0.0000e+00 - val_loss: 8.4245
Epoch 5/51
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 319ms/step - accuracy: 0.9946 - loss: 0.0277 - val_accuracy: 0.2344 - val_loss: 5.5620
Epoch 6/51
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 0.0096 - val_accuracy: 0.2500 - val_loss: 7.6702
Epoch 7/51
10/10 ━━━━━━━━━━━━━━━━━━━━ 11s 337ms/step - accuracy: 0.9969 - loss: 0.0183 - val_accuracy: 0.2500 - val_loss: 5.1993
Epoch 8/51
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 0.0105 - val_accuracy: 0.2500 - val_loss: 3.0230
Epoch 9/51
10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 366ms/step - accuracy: 0.9931 - loss: 0.0143 - val_accuracy: 0.2500 -

## **Video Classification**

In [ ]:
import numpy as np
import cv2
from keras._tf_keras.keras.applications import MobileNet
from keras._tf_keras.keras.preprocessing.image import load_img, img_to_array
from keras._tf_keras.keras.applications.mobilenet import preprocess_input
from keras._tf_keras.keras.models import Model
from keras._tf_keras.keras.layers import Dense, GlobalAveragePooling2D
from keras._tf_keras.keras.optimizers import Adam
from keras._tf_keras.keras.metrics import categorical_crossentropy
import tensorflow as tf
from keras._tf_keras.keras.preprocessing import image
from keras._tf_keras.keras.applications import imagenet_utils
import matplotlib.pyplot as plt

import os
from google.colab import drive
drive.mount('/content/drive')
video_links = []


base_url = '/content/drive/MyDrive/Hackathon/DataSet/Video Data/'

class_names = []
for class_name in os.listdir(base_url):
    class_dir = os.path.join(base_url, class_name)
    class_names.append(class_name)
    for filename in os.listdir(class_dir):
        link = f'{class_dir}/{filename}'
        video_links.append(link)

video_links = video_links[::-1]
class_names = class_names[::-1]

num_classes = len(video_links)



def video_frame_generator(video_path, num_frames):
  cap = cv2.VideoCapture(video_path)
  frames = []
  while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
      break
    frames.append(frame)
    if len(frames) == num_frames:
        break
    cap.release()
  return frames

def preprocess_data(video_path, num_frames):
  frames = video_frame_generator(video_path, num_frames)
  X = []
  for frame in frames:
    img = cv2.resize(frame, (224, 224 ))
    img = img_to_array(img)
    img = preprocess_input(img)
    X.append(img)
  X = np.array(X)
  return X
base_model = MobileNet(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
x = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=x)

model.compile(optimizer=Adam(learning_rate=0.001), loss=categorical_crossentropy, metrics=['accuracy'])

train_videos =  [ links  for links in video_links]
train_labels = [ i for i in range(0 , len(train_videos)) ]

X_train = []
y_train = []
for video_path, label in zip(train_videos, train_labels):
  X = preprocess_data(video_path, 10)

  X_train.extend(X)
  y_train.extend([label] * X.shape[0])

X_train = np.array(X_train)
y_train = np.array(y_train)


train_labels = [label % num_classes for label in train_labels]  # Adjust labels if necessary



y_train = tf.keras.utils.to_categorical(y_train, num_classes=num_classes) # One-hot encode labels



history = model.fit(X_train, y_train, epochs=10, batch_size=32)

model.save('video_classification_model.h5')

# test_video_path = '/content/drive/MyDrive/Hackathon/DataSet/Video Data/About/about.mp4'

# X_test = preprocess_data(test_video_path, 10)


# predictions = model.predict(X_test)

# predicted_class_index = np.argmax(predictions[0])  # Get the index of the class with the highest probability

# class_labels = [label for label in os.listdir(base_url)]

# predicted_class_label = class_labels[predicted_class_index]

# if predicted_class_label is not None:
#   print(f"Predicted class: {predicted_class_label}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - accuracy: 0.0500 - loss: 3.2435
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 0.7000 - loss: 2.0154
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.8500 - loss: 1.2128
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 1.0000 - loss: 0.6992
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.9000 - loss: 0.5010
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step - accuracy: 1.0000 - loss: 0.2913
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 1.0000 - loss: 0.1993
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 1.0000 - loss: 0.1440
Epoch 9/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 1.0000 - loss: 0.0932
Epoch 10/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 1.0000 - loss: 0.0670


## **UI for Model**

In [ ]:
!pip install gradio
!pip install python-multipart


In [ ]:
import gradio as gr
import os
import numpy as np
import tensorflow as tf
from keras._tf_keras.keras.models import load_model
from keras._tf_keras.keras.preprocessing.image import img_to_array
from keras._tf_keras.keras.applications.mobilenet import preprocess_input
import cv2


def classify_video(video_path):
  model = load_model('video_classification_model.h5')
  X_test = preprocess_data(video_path, 10)
  predictions = model.predict(X_test)
  predicted_class_index = np.argmax(predictions[0])
  class_labels = [label for label in os.listdir(base_url)]
  class_labels = class_labels[::-1]
  predicted_class_label = class_labels[predicted_class_index]
  return predicted_class_label

video_iface = gr.Interface(
    fn=classify_video,
    inputs=gr.Video(),
    outputs="text",
    title="Video Classification",
    description="Upload a video to classify it.",
)

demo = gr.TabbedInterface([video_iface], ["Video Classification"])

demo.launch()

Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e8ebcafc50812d270d.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
